In [5]:
import os
import json
from dotenv import load_dotenv
import gradio as gr
from bardapi import Bard
import google.generativeai as genai
import gradio as gr

In [39]:
# Load biến môi trường từ file .env
load_dotenv(override=True)

# Lấy API key
bard_api_key = os.getenv('BARD_API_KEY')

# Gán key và khởi tạo Bard nếu có
if bard_api_key:
    os.environ['_BARD_API_KEY'] = bard_api_key
    bard = Bard()
    print(f"Bard API Key loaded and begins with: {bard_api_key[:8]}")
else:
    print("Bard API Key not set")


genai.configure(api_key=bard_api_key)
model = genai.GenerativeModel("gemini-1.5-pro")

Bard API Key loaded and begins with: AIzaSyB6


In [ ]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [13]:
system_message = "Bạn là một trợ lý AI hữu ích."

def chat(message, history):
    prompt = f"{system_message}\n"
    
    for msg in history:
        role = msg['role'].capitalize()
        content = msg['content']
        if role == 'User':
            prompt += f"User: {content}\n"
        elif role == 'Assistant':
            prompt += f"Assistant: {content}\n"
        else:
            prompt += f"System: {content}\n"  # Nếu là hệ thống (nếu cần thiết)

    prompt += f"User: {message}\nAssistant:"

    model = genai.GenerativeModel("gemini-1.5-pro")
    stream = model.generate_content(prompt, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.text or ''
        yield response

# Tạo giao diện Gradio
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7905

To create a public link, set `share=True` in `launch()`.


In [15]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

get_ticket_price("London")

Tool get_ticket_price called for London


'$799'

In [17]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]

In [ ]:
def chat1(message, history):
    system_message = "Bạn là một trợ lý AI hữu ích."
    
    prompt = f"{system_message}\n"
    for msg in history:
        role = msg['role'].capitalize()
        content = msg['content']
        if role == 'User':
            prompt += f"User: {content}\n"
        elif role == 'Assistant':
            prompt += f"Assistant: {content}\n"
        else:
            prompt += f"System: {content}\n"

    prompt += f"User: {message}\nAssistant:"
    stream = model.generate_content(prompt, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.text or ''
        yield response
    
    if "tool_calls" in response:
        message = response['choices'][0]['message']
        response, city = handle_tool_call(message)  # Hàm xử lý công cụ được gọi
        
        messages = history + [{"role": "user", "content": message}]
        messages.append({"role": "assistant", "content": response['content']})
        
        response = model.generate_content("\n".join([msg['content'] for msg in messages]), stream=True)
    
    return response['choices'][0]['message']['content']


In [ ]:
def handle_tool_call(message):
    if 'content' in message and isinstance(message['content'], dict) and 'destination_city' in message['content']:
        city = message['content']['destination_city']
        price = get_ticket_price(city)
        
        response = {
            "role": "assistant",
            "content": f"The ticket price to {city} is {price}.",
        }
        return response, city
    else:
        response = {
            "role": "assistant",
            "content": "Sorry, I couldn't find the destination city in the message."
        }
        return response, None


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [22]:
import base64
from io import BytesIO
from PIL import Image

In [43]:
def generate_city_description(city):
    prompt = f"Create a vivid description of a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style."
    
    response = model.generate_content(prompt)  # Đảm bảo model.generate_content hoạt động đúng
    description = response.text  # Đảm bảo bạn sử dụng đúng cú pháp của đối tượng trả về
    return description

# Hàm tạo hình ảnh từ mô tả, với việc xử lý lỗi khi kết nối API
def generate_image_from_description(description):
    try:
        image_response = requests.post(
            "https://api.imagen.ai/generate",
            headers={"Authorization": f"Bearer {os.getenv('IMAGEN_API_KEY')}"},
            json={"prompt": description, "size": "1024x1024"}
        )

        # Kiểm tra nếu API trả về lỗi
        image_response.raise_for_status()

        image_data = image_response.content
        return Image.open(BytesIO(image_data))
    except requests.exceptions.RequestException as e:
        print(f"Lỗi khi tạo hình ảnh: {e}")
        return None

# Hàm chính kết hợp cả mô tả và hình ảnh
def artist(city):
    description = generate_city_description(city)
    print("Description:", description)
    
    # Dùng mô tả để tạo hình ảnh
    image = generate_image_from_description(description)
    if image:
        return image
    else:
        print("Không thể tạo hình ảnh.")
        return None

# Ví dụ: tạo hình ảnh cho New York City
image = artist("New York City")
if image:
    image.show()

Description: BAM!  New York City exploded in a kaleidoscope of yellow cabs and fuchsia lipstick kisses. Think Lichtenstein dots peppering Times Square, a Warholian Marilyn Monroe morphing into Lady Liberty’s emerald face.  My vacation? POW! A sensory overload served on a silver platter of skyscrapers.

First stop, the Empire State Building –  KA-BOOM! –  a colossal Art Deco rocket blasting into a cerulean sky.  The city sprawled below, a Mondrian grid of streets vibrating with energy. Tiny yellow taxis zipped like bumblebees across the canvas, a blur of chrome and sunshine.

Next, Central Park.  WHAM! An explosion of green in a concrete jungle.  Horse-drawn carriages, a whimsical touch of old-world charm, clip-clopped past street performers juggling neon hula hoops. Hot dog vendors, their carts a riot of primary colors, hawked their wares, the scent of mustard a pungent punctuation mark in the crisp autumn air.

The Metropolitan Museum of Art: SWISH! A whirlwind tour through centuries 